In [ ]:
# Cell 1: Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import pickle
import os

# Cell 2: Load Data
df = pd.read_csv('../data/spotify_songs_pool.csv')
print(f"Loaded database shape: {df.shape}")

# Separate numeric audio features from descriptive metadata
X = df.drop(columns=['id', 'song', 'artist'])
print("Features used for clustering:", X.columns.tolist())

# Cell 3: Scale Features
# Essential step for distance-based models like K-Means
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

# Cell 4: The Elbow Method & Silhouette Score to determine 'K'
K = range(2, 15)
inertia = []
silhouette = []

for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)
    silhouette.append(silhouette_score(X_scaled, kmeans.labels_))

# Plot the Elbow Curve
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(K, inertia, 'bx-')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('The Elbow Method')

# Plot Silhouette Scores
plt.subplot(1, 2, 2)
plt.plot(K, silhouette, 'rx-')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Scores')

plt.tight_layout()
plt.show()

# Cell 5: Train Final K-Means Model
# Based on the plots, select the best k (let's assume 8 for the MVP)
optimal_k = 8 
final_kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
final_kmeans.fit(X_scaled)

# Append cluster assignments back to original dataframe
df['cluster'] = final_kmeans.labels_
print(df['cluster'].value_counts().sort_index())

# Cell 6: Save Outputs for the Production App
os.makedirs('../models', exist_ok=True)

# Save the configured DataFrame
df.to_csv('../data/clustered_songs.csv', index=False)

# Pickle the scaler and model so our production script can transform user data on the fly
with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('../models/kmeans_model.pkl', 'wb') as f:
    pickle.dump(final_kmeans, f)

print("Saved model artifacts and final clustered database successfully!")